# Notebook 12: Soft Actor-Critic (SAC)

## Learning Goals

By the end of this notebook you will be able to:

1. **Explain** the Maximum Entropy RL framework and why entropy bonuses encourage exploration.
2. **Describe** the Reparameterization Trick and how it enables gradient-based policy optimization.
3. **Understand** why Twin Critics (Clipped Double-Q) reduce Q-value overestimation.
4. **Implement** the Polyak (soft) target update rule.
5. **Train** a SAC agent on the InvertedPendulum and compare its performance with DQN (notebook 10) and PPO (notebook 11).

---

## Prerequisites

| Notebook | Topic |
|----------|-------|
| 09 | Actor-Critic (A2C) |
| 10 | Deep Q-Network (DQN, Experience Replay, Target Network) |
| 11 | PPO (Clipped Surrogate, GAE) |

## 1. Motivation: Why Maximum Entropy RL?

Standard RL maximises cumulative reward:

$$J_\text{std}(\pi) = \sum_{t} \mathbb{E}[r(s_t, a_t)]$$

**Soft** RL adds an entropy bonus:

$$J_\text{SAC}(\pi) = \sum_{t} \mathbb{E}\bigl[r(s_t, a_t) + \alpha \, \mathcal{H}(\pi(\cdot|s_t))\bigr]$$

where $\mathcal{H}(\pi) = -\mathbb{E}_{a \sim \pi}[\log \pi(a|s)]$ is the policy entropy and $\alpha > 0$ is the **temperature** parameter.

### Benefits of entropy maximisation

| Without entropy bonus | With entropy bonus |
|---|---|
| Collapses to a deterministic policy | Maintains stochasticity; explores broadly |
| Sensitive to reward scaling | More robust to reward shaping |
| May get stuck in local optima | Naturally avoids premature convergence |

SAC combines this idea with **off-policy learning** (replay buffer) and **continuous action spaces**, making it one of the most data-efficient deep RL algorithms.

## 2. SAC Key Concepts

### 2.1 Soft Bellman Equation

The soft Q-function satisfies:

$$Q^\pi(s,a) = r + \gamma \, \mathbb{E}_{s' \sim p}\bigl[V^\pi(s')\bigr]$$

$$V^\pi(s) = \mathbb{E}_{a \sim \pi}\bigl[Q^\pi(s,a) - \alpha \log \pi(a|s)\bigr]$$

Combining them gives the **SAC TD target**:

$$y = r + \gamma \bigl(\min_j Q_{\theta^-_j}(s', \tilde{a}') - \alpha \log \pi(\tilde{a}'|s')\bigr)$$

where $\tilde{a}' \sim \pi(\cdot|s')$ is sampled from the **current** policy.

### 2.2 Twin Critics (Clipped Double-Q)

Two independent Q-networks $Q_{\theta_1}$, $Q_{\theta_2}$ are trained in parallel. The TD target uses the **minimum** of both:

$$\min(Q_{\theta^-_1}(s',\tilde{a}'), \, Q_{\theta^-_2}(s',\tilde{a}'))$$

This **under-estimates** rather than over-estimates Q, counteracting the maximisation bias.

### 2.3 Reparameterization Trick

We use a Gaussian policy with a **squashing** function:

$$a = \text{tanh}(\mu_\theta(s) + \sigma_\theta(s) \cdot \varepsilon), \quad \varepsilon \sim \mathcal{N}(0, I)$$

The log-probability of the squashed action is:

$$\log \pi(a|s) = \log \mathcal{N}(u | \mu, \sigma^2) - \sum_i \log(1 - \tanh^2(u_i))$$

where $u$ is the pre-squash action.

### 2.4 Polyak Soft Target Update

Instead of a hard copy (DQN), target networks are updated **gradually**:

$$\theta^- \leftarrow \tau \theta + (1-\tau) \theta^-$$

A small $\tau$ (e.g. 0.005) ensures stable TD targets.

### 2.5 Automatic Temperature Tuning

Instead of hand-tuning $\alpha$, we treat it as a learnable parameter:

$$\mathcal{L}(\alpha) = -\alpha \cdot \bigl(\log \pi(\tilde{a}|s) + \mathcal{H}_{\text{target}}\bigr)$$

where $\mathcal{H}_{\text{target}} \approx -\dim(\mathcal{A})$ is a desired minimum entropy.

## 3. SAC vs. Previous Algorithms — Quick Comparison

| Feature | Q-learning (07) | DQN (10) | PPO (11) | **SAC (12)** |
|---|---|---|---|---|
| Action space | Discrete | Discrete | Continuous | **Continuous** |
| On / Off-policy | On | Off | On | **Off** |
| Replay buffer | No | Yes | No | **Yes** |
| Target network | No | Hard copy | — | **Polyak** |
| Entropy bonus | No | No | Implicit (clip) | **Explicit** |
| # of networks | 2 (Q, Q-target) | 2 | 1 (Actor-Critic) | **5** (Actor, Q1, Q2, Q1-target, Q2-target) |

In [ ]:
# Setup
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

import torch
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')

from src.environments import InvertedPendulumEnv
from src.policies.sac_policy import SACPolicy, SACActorNet, SACCriticNet, SACReplayBuffer

print('Imports OK')

## 4. Network Architecture Walkthrough

In [ ]:
# --- Actor network ---
actor = SACActorNet(state_dim=4, action_dim=1, hidden_sizes=[256, 256])
print('=== Actor (Gaussian policy) ===')
print(actor)
n_actor = sum(p.numel() for p in actor.parameters())
print(f'Parameters: {n_actor:,}')

# Sample a state and run a forward pass
dummy_state = torch.FloatTensor([[0.0, 0.0, 0.1, 0.0]])
mean, log_std = actor(dummy_state)
print(f'\nOutput for state={dummy_state.numpy().ravel()}:')
print(f'  mean    = {mean.item():.4f}')
print(f'  log_std = {log_std.item():.4f}  ->  std = {log_std.exp().item():.4f}')

In [ ]:
# --- Critic network ---
critic = SACCriticNet(state_dim=4, action_dim=1, hidden_sizes=[256, 256])
print('=== Critic Q(s,a) ===')
print(critic)
n_critic = sum(p.numel() for p in critic.parameters())
print(f'Parameters: {n_critic:,}')

dummy_action = torch.FloatTensor([[5.0]])
q_val = critic(dummy_state, dummy_action)
print(f'\nQ(s, a=5.0) = {q_val.item():.4f}')

## 5. Visualise Clipping Effect — Reparameterization

Let's visualise how the `tanh` squashing affects the action distribution.

In [ ]:
# Show how tanh squashes a Gaussian
np.random.seed(42)
mu, sigma = 0.0, 1.0
u_samples = np.random.normal(mu, sigma, size=10_000)
a_samples = np.tanh(u_samples)

action_scale = 10.0
scaled_actions = a_samples * action_scale

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(u_samples, bins=60, color='steelblue', alpha=0.8)
axes[0].set_title('Pre-squash u ~ N(0,1)')
axes[0].set_xlabel('u')
axes[0].set_ylabel('Count')

axes[1].hist(a_samples, bins=60, color='darkorange', alpha=0.8)
axes[1].set_title('After tanh(u) in (-1, 1)')
axes[1].set_xlabel('tanh(u)')

axes[2].hist(scaled_actions, bins=60, color='forestgreen', alpha=0.8)
axes[2].set_title('Scaled action = tanh(u) x 10')
axes[2].set_xlabel('Force (N)')
axes[2].axvline(-10, color='red', linestyle='--', linewidth=1.5, label='bounds')
axes[2].axvline( 10, color='red', linestyle='--', linewidth=1.5)
axes[2].legend()

plt.suptitle('Reparameterization Trick: Gaussian -> Squashed Action', fontsize=13)
plt.tight_layout()
plt.show()

The `tanh` squashing ensures actions always stay within $[-10, 10]$ while the policy remains a smooth, differentiable distribution.

## 6. Training Loop

In [ ]:
def train_sac(
    env,
    policy: SACPolicy,
    n_episodes: int = 200,
    update_every: int = 1,
    verbose: bool = True,
):
    """
    SAC training loop.

    SAC is off-policy: we collect one step at a time and call update()
    every `update_every` environment steps.

    Returns
    -------
    history : dict with lists of per-episode scalars
    """
    history = {
        'episode_reward': [],
        'critic1_loss': [],
        'critic2_loss': [],
        'actor_loss': [],
        'alpha': [],
    }

    global_step = 0

    for ep in range(1, n_episodes + 1):
        state = env.reset()
        ep_reward = 0.0
        ep_c1, ep_c2, ep_actor, ep_alpha = [], [], [], []
        done = False

        while not done:
            action = policy.get_action_train(state)
            next_state, reward, done, _ = env.step(action)
            policy.push(state, action, reward, next_state, done)
            state = next_state
            ep_reward += reward
            global_step += 1

            if global_step % update_every == 0:
                stats = policy.update()
                if stats is not None:
                    ep_c1.append(stats['critic1_loss'])
                    ep_c2.append(stats['critic2_loss'])
                    ep_actor.append(stats['actor_loss'])
                    ep_alpha.append(stats['alpha'])

        history['episode_reward'].append(ep_reward)
        history['critic1_loss'].append(np.mean(ep_c1) if ep_c1 else float('nan'))
        history['critic2_loss'].append(np.mean(ep_c2) if ep_c2 else float('nan'))
        history['actor_loss'].append(np.mean(ep_actor) if ep_actor else float('nan'))
        history['alpha'].append(np.mean(ep_alpha) if ep_alpha else policy.alpha)

        if verbose and ep % 20 == 0:
            recent_reward = np.mean(history['episode_reward'][-20:])
            alpha_val = history['alpha'][-1]
            print(f'Ep {ep:4d} | Avg Reward (last 20): {recent_reward:7.1f} | alpha: {alpha_val:.4f}')

    return history

In [ ]:
# Create environment and SAC policy
env = InvertedPendulumEnv(max_steps=500)

sac_policy = SACPolicy(
    state_dim=4,
    action_dim=1,
    action_scale=10.0,
    action_bias=0.0,
    hidden_sizes=[256, 256],
    lr=3e-4,
    gamma=0.99,
    tau=0.005,
    auto_alpha=True,
    batch_size=256,
    buffer_capacity=100_000,
    warmup_steps=1_000,   # random exploration before training
)

print(sac_policy)
print(f'Total parameters: {sac_policy.get_num_params():,}')

In [ ]:
# Train SAC
print('Training SAC on InvertedPendulum...')
print('(warmup phase: 1000 random steps, then learning begins)')
print()

history = train_sac(env, sac_policy, n_episodes=200, update_every=1, verbose=True)

## 7. Learning Curves

In [ ]:
def smooth(values, window=10):
    """Moving average smoothing."""
    result = []
    for i in range(len(values)):
        start = max(0, i - window + 1)
        result.append(np.nanmean(values[start:i+1]))
    return result

episodes = np.arange(1, len(history['episode_reward']) + 1)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Episode reward
ax = axes[0, 0]
ax.plot(episodes, history['episode_reward'], alpha=0.3, color='steelblue', label='raw')
ax.plot(episodes, smooth(history['episode_reward'], 20), color='steelblue', linewidth=2, label='smoothed (20)')
ax.set_xlabel('Episode')
ax.set_ylabel('Total Reward')
ax.set_title('Episode Reward')
ax.legend()
ax.grid(alpha=0.3)

# Critic losses
ax = axes[0, 1]
ax.plot(episodes, smooth(history['critic1_loss'], 20), label='Critic 1 loss', color='darkorange')
ax.plot(episodes, smooth(history['critic2_loss'], 20), label='Critic 2 loss', color='tomato', linestyle='--')
ax.set_xlabel('Episode')
ax.set_ylabel('MSE Loss')
ax.set_title('Critic Losses (TD Error)')
ax.legend()
ax.grid(alpha=0.3)

# Actor loss
ax = axes[1, 0]
ax.plot(episodes, smooth(history['actor_loss'], 20), color='forestgreen', linewidth=2)
ax.set_xlabel('Episode')
ax.set_ylabel('Actor Loss')
ax.set_title('Actor Loss (Q - alpha * log pi)')
ax.grid(alpha=0.3)

# Temperature alpha
ax = axes[1, 1]
ax.plot(episodes, history['alpha'], color='purple', linewidth=2)
ax.set_xlabel('Episode')
ax.set_ylabel('Alpha (temperature)')
ax.set_title('Auto-tuned Temperature Alpha')
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.grid(alpha=0.3)

plt.suptitle('SAC Training Curves — InvertedPendulum', fontsize=14)
plt.tight_layout()
plt.show()

## 8. Evaluation

In [ ]:
def evaluate_policy(env, policy, n_episodes=20):
    """Evaluate using deterministic (greedy) actions."""
    rewards = []
    for _ in range(n_episodes):
        state = env.reset()
        total_reward = 0.0
        done = False
        while not done:
            action = policy.get_action(state)   # deterministic mean
            state, reward, done, _ = env.step(action)
            total_reward += reward
        rewards.append(total_reward)
    return {
        'mean': np.mean(rewards),
        'std': np.std(rewards),
        'min': np.min(rewards),
        'max': np.max(rewards),
        'rewards': rewards,
    }

eval_result = evaluate_policy(env, sac_policy, n_episodes=20)
print('=== SAC Evaluation (20 episodes, deterministic) ===')
print(f'Mean reward : {eval_result["mean"]:7.1f}')
print(f'Std         : {eval_result["std"]:7.1f}')
print(f'Min / Max   : {eval_result["min"]:7.1f} / {eval_result["max"]:7.1f}')

In [ ]:
# Visualise evaluation distribution
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(eval_result['rewards'], bins=15, color='steelblue', alpha=0.8, edgecolor='white')
ax.axvline(eval_result['mean'], color='red', linewidth=2, label=f'Mean={eval_result["mean"]:.1f}')
ax.set_xlabel('Episode Total Reward')
ax.set_ylabel('Count')
ax.set_title('SAC Evaluation — Reward Distribution (20 episodes)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Comparison with DQN and A2C

Let's compare the convergence of SAC versus the simpler A2C (Actor-Critic) from notebook 09.

In [ ]:
from src.policies.actor_critic import ActorCriticPolicy

def train_a2c(env, n_episodes=200):
    """Train Actor-Critic (A2C) for comparison."""
    policy = ActorCriticPolicy(hidden_sizes=[64, 64], gamma=0.99)
    rewards = []
    for ep in range(1, n_episodes + 1):
        state = env.reset()
        done = False
        while not done:
            action, log_prob, value = policy.get_action_train(state)
            next_state, reward, done, _ = env.step(action)
            policy.store_transition(log_prob, reward, value)
            state = next_state
        result = policy.update_on_episode()
        rewards.append(result['total_reward'])
    return rewards

print('Training A2C for comparison...')
a2c_env = InvertedPendulumEnv(max_steps=500)
a2c_rewards = train_a2c(a2c_env, n_episodes=200)
print('Done!')

In [ ]:
episodes = np.arange(1, 201)

fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(episodes, smooth(a2c_rewards, 20),
        label='A2C (on-policy)', color='darkorange', linewidth=2, alpha=0.9)
ax.plot(episodes, smooth(history['episode_reward'], 20),
        label='SAC (off-policy + entropy)', color='steelblue', linewidth=2, alpha=0.9)

ax.set_xlabel('Episode')
ax.set_ylabel('Total Reward (smoothed, window=20)')
ax.set_title('SAC vs. A2C — InvertedPendulum (200 episodes)')
ax.legend(fontsize=12)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Final 20-ep average — A2C : {np.mean(a2c_rewards[-20:]):.1f}')
print(f'Final 20-ep average — SAC : {np.mean(history["episode_reward"][-20:]):.1f}')

## 10. Effect of the Temperature Alpha

Let's run two fixed-alpha variants to see how $\alpha$ affects exploration.

In [ ]:
results = {}
for alpha_val in [0.05, 0.5]:
    print(f'Training SAC with fixed alpha={alpha_val}...')
    env_tmp = InvertedPendulumEnv(max_steps=500)
    pol_tmp = SACPolicy(
        hidden_sizes=[256, 256],
        auto_alpha=False,
        alpha=alpha_val,
        batch_size=256,
        buffer_capacity=50_000,
        warmup_steps=500,
    )
    h_tmp = train_sac(env_tmp, pol_tmp, n_episodes=150, verbose=False)
    results[alpha_val] = h_tmp['episode_reward']
    print(f'  Final avg: {np.mean(results[alpha_val][-20:]):.1f}')

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['darkorange', 'steelblue']
for (alpha_val, rew), col in zip(results.items(), colors):
    eps = np.arange(1, len(rew) + 1)
    ax.plot(eps, smooth(rew, 20), label=f'alpha={alpha_val} (fixed)', color=col, linewidth=2)

ax.set_xlabel('Episode')
ax.set_ylabel('Reward (smoothed)')
ax.set_title('Effect of Fixed Temperature Alpha on SAC Performance')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 11. Key Takeaways

1. **Maximum Entropy RL** adds $\alpha \mathcal{H}(\pi)$ to the objective, encouraging exploration and robustness.

2. **Reparameterization Trick** allows gradient flow through the stochastic sampling step, enabling low-variance policy gradient updates.

3. **Twin Critics** (Clipped Double-Q) reduce Q-value overestimation by taking $\min(Q_1, Q_2)$ in the TD target.

4. **Polyak soft target updates** ($\tau = 0.005$) provide more stable TD targets than hard copies.

5. **Automatic temperature tuning** removes the need to hand-tune $\alpha$ by optimising it with a target entropy constraint.

6. SAC is **off-policy**, so it reuses data efficiently via the replay buffer — often more sample-efficient than on-policy methods like A2C or PPO.

---

## 12. What's Next?

| Topic | Notebook |
|---|---|
| Multi-Agent RL basics (prisoner's dilemma, IQL) | 13 |
| Cooperative MARL (CTDE pattern) | 14 |

---

## Practice Problems

1. **Replay buffer size**: Halve `buffer_capacity` to 50,000. Does SAC still converge? Why might a larger buffer help?

2. **Warmup steps**: Remove the warmup phase (`warmup_steps=0`). What happens to the early training behaviour?

3. **Target entropy**: Modify `target_entropy` to $-0.5$ (less exploration) and $-2.0$ (more). Compare learning curves.

4. **Network size**: Try `hidden_sizes=[64, 64]` vs. the default `[256, 256]`. How does capacity affect final performance?

5. **SAC vs DQN**: SAC operates in continuous action space. If you discretised the actions (like DQN does), what would you lose?